Prédiction du Taux de Pauvreté à l'Aide de la Régression Logistique avec Réduction de Dimensions par PCA

Ce script réalise une analyse détaillée pour prédire une variable binaire (pcexp_binaire) en utilisant la régression logistique avec réduction de dimensions par PCA. Voici une explication détaillée de ses étapes :
1. Chargement et Préparation des Données
Le script charge un fichier CSV contenant des données riches en variables (4096 fonctionnalités extraites) et des colonnes cibles. Il sélectionne les colonnes des fonctionnalités (feature_0 à feature_4095) comme prédicteurs et les variables cibles pcexp_binaire (binaire) et log_pcexp (continue) pour l'analyse.
Ensuite, il divise les données en ensembles d'entraînement (60%) et de test (40%) avec une graine aléatoire pour assurer la reproductibilité. Les données sont ensuite standardisées pour garantir une mise à l'échelle uniforme des variables, essentielle pour le fonctionnement optimal du PCA et de la régression logistique.
2. Réduction de Dimensions avec PCA
Le script applique une Analyse en Composantes Principales (PCA) pour réduire la dimensionnalité, en conservant suffisamment de composantes pour expliquer 97% de la variabilité des données. Cela réduit le bruit et la redondance dans les prédicteurs tout en diminuant la complexité computationnelle. Le nombre de composantes retenues est affiché.
3. Entraînement du Modèle de Régression Logistique
Un modèle de régression logistique est ajusté sur les données d’entraînement transformées par PCA. Une pondération des classes (class_weight='balanced') est utilisée pour compenser le déséquilibre entre les classes (plus de 0 que de 1 dans pcexp_binaire). Le solveur saga est employé avec une limite de 500 itérations pour l’optimisation.
Le modèle est ensuite utilisé pour prédire les classes et les probabilités sur l’ensemble de test.
4. Sauvegarde du Modèle
Le modèle entraîné est sauvegardé sous forme de fichier pickle dans le répertoire spécifié, ce qui permet une réutilisation ultérieure sans nécessiter de réentraînement.
5. Évaluation des Performances
Le script évalue les performances du modèle à l’aide des métriques suivantes :
- Matrice de confusion : Elle est calculée et affichée sous forme textuelle et graphique (avec Seaborn).
- Rapport de classification : Il inclut des métriques telles que la précision, le rappel et le F1-score.
- Accuracy, précision et spécificité : Ces métriques sont calculées à partir de la matrice de confusion pour évaluer la qualité globale des prédictions.
6. Visualisation de la Courbe ROC
Une courbe ROC (Receiver Operating Characteristic) est tracée pour illustrer la performance du modèle à différents seuils de classification, avec l’AUC (Area Under Curve) affiché pour quantifier cette performance.
7. Calcul du Taux de Pauvreté
Le script calcule deux taux de pauvreté :
- Taux réel : Basé sur la variable binaire pcexp_binaire réelle, pondérée par la taille des ménages et leur poids (hhsize, hhweight).
- Taux prédit : Basé sur les prédictions du modèle (pcexp_binaire_pred), également pondérées.
Ces taux sont affichés et sauvegardés dans un fichier CSV, qui inclut les prédictions pour chaque observation de l’ensemble de test.
8. Sauvegarde des Résultats
Les résultats finaux, incluant les prédictions et les taux de pauvreté calculés, sont exportés dans un fichier CSV. Les figures générées (matrice de confusion et courbe ROC) sont également sauvegardées dans un répertoire dédié.

In [ ]:
import pandas as pd

# Chemin du fichier d'entrée
input_csv = r"D:\wealth_predict_sentinel\Data\processed_csv\EHCVM_2018_images_with_features_extracted_4096_fullyConv_sans_augm_couche_nongelee_batch_16_version_2.csv"

# Chargement des données
df = pd.read_csv(input_csv)

# Vérifier si la colonne 'pcexp_binaire' existe déjà
if 'pcexp_binaire' not in df.columns:
    # Vérifier si les colonnes nécessaires existent
    if 'pcexp' in df.columns and 'zref' in df.columns:
        # Ajouter la colonne 'pcexp_binaire'
        df['pcexp_binaire'] = (df['pcexp'] <= df['zref']).astype(int)
        print("La colonne 'pcexp_binaire' a été ajoutée avec succès.")
    else:
        raise ValueError("Les colonnes 'pcexp' et/ou 'zref' sont manquantes dans le fichier.")

# Sauvegarde du fichier CSV mis à jour
df.to_csv(input_csv, index=False)
print(f"Fichier mis à jour et sauvegardé : {input_csv}")


In [ ]:
!pip install seaborn

In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import seaborn as sns
import matplotlib.pyplot as plt
import os
import pickle

# Configuration des chemins
input_csv = r"D:\wealth_predict_sentinel\Data\processed_csv\EHCVM_2018_images_with_features_extracted_4096_fullyConv_sans_augm_couche_nongelee_batch_16_version_2.csv"
output_dir_csv = r"D:\wealth_predict_sentinel\Data\processed_csv"
output_dir_models = r"D:\wealth_predict_sentinel\models"
output_dir_figures = r"D:\wealth_predict_sentinel\figures_graphs"

# Chemins pour sauvegarder le StandardScaler et le PCA
scaler_path = os.path.join(output_dir_models, "scaler_fullyConv_sans_augm_couche_nongelee_batch_16_regular_split_version_3.pkl")
pca_path = os.path.join(output_dir_models, "pca_fullyConv_sans_augm_couche_nongelee_batch_16_regular_split_version_3.pkl")

# Chargement des données
df = pd.read_csv(input_csv)
features = [f'feature_{i}' for i in range(4096)]
target_binary = 'pcexp_binaire'

X = df[features]
y_binary = df[target_binary]

# Étape 1 : Division des données
X_train, X_test, y_train, y_test = train_test_split(X, y_binary, test_size=0.7, random_state=42)

# Standardisation des données
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Sauvegarde du StandardScaler
with open(scaler_path, 'wb') as file:
    pickle.dump(scaler, file)

print(f"Le StandardScaler a été sauvegardé sous : {scaler_path}")

# Étape 2 : Réduction de dimension avec PCA
pca = PCA(n_components=0.999)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
num_pca_components = X_train_pca.shape[1]
print(f"Nombre de composantes principales expliquant 99.9% de la variabilité : {num_pca_components}")

# Sauvegarde du modèle PCA
with open(pca_path, 'wb') as file:
    pickle.dump(pca, file)

print(f"Le modèle PCA a été sauvegardé sous : {pca_path}")

# Étape 3 : Régression logistique
logistic_model = LogisticRegression(class_weight='balanced', max_iter=5000, solver='saga', random_state=42)
logistic_model.fit(X_train_pca, y_train)

# Prédictions
y_train_pred = logistic_model.predict(X_train_pca)
y_test_pred = logistic_model.predict(X_test_pca)
y_test_proba = logistic_model.predict_proba(X_test_pca)[:, 1]

# Sauvegarde du modèle
logistic_model_path = os.path.join(output_dir_models, "fullyConv_sans_augm_couche_nongelee_batch_16_logistic_pca_version_3.pkl")
with open(logistic_model_path, 'wb') as file:
    pickle.dump(logistic_model, file)

print(f"Le modèle a été sauvegardé sous : {logistic_model_path}")

# Évaluation
print("Rapport de classification (Test Set) :")
print(classification_report(y_test, y_test_pred))

conf_matrix = confusion_matrix(y_test, y_test_pred)
print("Matrice de confusion (Test Set) :")
print(conf_matrix)

# Affichage de la confusion matrix avec seaborn
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=['Classe 0', 'Classe 1'], yticklabels=['Classe 0', 'Classe 1'])
plt.title("Matrice de confusion (Test Set) - Seaborn")
plt.xlabel("Prédictions")
plt.ylabel("Valeurs Réelles")
plt.savefig(os.path.join(output_dir_figures, "fullyConv_sans_augm_couche_nongelee_batch_16_confusion_matrix_logistic_reg_version_3.png"))
plt.show()

# Calcul des métriques
accuracy = (conf_matrix[0, 0] + conf_matrix[1, 1]) / conf_matrix.sum()
precision = conf_matrix[1, 1] / (conf_matrix[1, 1] + conf_matrix[0, 1])
specificity = conf_matrix[0, 0] / (conf_matrix[0, 0] + conf_matrix[0, 1])

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Specificity : {specificity:.4f}")

# Courbe ROC et AUC
fpr, tpr, _ = roc_curve(y_test, y_test_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'r--', label="Random Guessing")
plt.xlabel("Taux de faux positifs (FPR)")
plt.ylabel("Taux de vrais positifs (TPR)")
plt.title(f"ROC Curve pour Régression Logistique avec PCA")
plt.legend()
plt.grid(True)
roc_curve_path = os.path.join(output_dir_figures, "fullyConv_sans_augm_couche_nongelee_batch_16_roc_curve_logistic_reg_version_3.png")
plt.savefig(roc_curve_path)
plt.show()

# Étape 4 : Calcul du taux de pauvreté
df_test = df.loc[X_test.index]
df_test['pcexp_binaire_pred'] = y_test_pred

# Calcul du taux de pauvreté réel
weighted_real_poverty = (
    (df_test['hhweight'] * df_test['hhsize'] * df_test[target_binary]).sum() /
    (df_test['hhweight'] * df_test['hhsize']).sum()
)

# Calcul du taux de pauvreté prédit
weighted_predicted_poverty = (
    (df_test['hhweight'] * df_test['hhsize'] * df_test['pcexp_binaire_pred']).sum() /
    (df_test['hhweight'] * df_test['hhsize']).sum()
)

print(f"Taux de pauvreté réel : {weighted_real_poverty:.4%}")
print(f"Taux de pauvreté prédit : {weighted_predicted_poverty:.4%}")

# Sauvegarde des résultats finaux
final_csv_path = os.path.join(output_dir_csv, "fullyConv_sans_augm_couche_nongelee_batch_16_test_with_poverty_logistic_reg_version_3.csv")
df_test.to_csv(final_csv_path, index=False)

print(f"Les résultats ont été sauvegardés dans : {final_csv_path}")


Prédiction des Classes Binaires avec Réduction de Dimension par PCA et Validation Croisée k-Fold Utilisant une Régression Logistique

Ce script effectue une analyse approfondie pour prédire des classes binaires (0 ou 1) en utilisant une régression logistique avec réduction de dimension via PCA et validation croisée k-fold. Voici une description détaillée de ses étapes principales :
Chargement et Préparation des Données
Le script commence par charger un fichier CSV contenant des caractéristiques extraites (feature_0 à feature_4095) et une cible binaire (pcexp_binaire) représentant si une observation est au-dessus ou en dessous d'un seuil de pauvreté. Les caractéristiques sont extraites pour créer les matrices X (features) et y (target binaire).
Standardisation des Données
Les données sont standardisées à l'aide de StandardScaler pour assurer que chaque caractéristique ait une moyenne de 0 et un écart-type de 1, ce qui est essentiel pour optimiser les performances des algorithmes tels que PCA et régression logistique.
Réduction de Dimension avec PCA
Le script applique une réduction de dimension avec PCA, en sélectionnant les composantes principales qui expliquent 98% de la variabilité des données. Cela réduit le nombre de variables à un ensemble plus compact et pertinent, tout en maintenant une grande partie de l'information initiale.
Régression Logistique avec Validation Croisée k-Fold
Un modèle de régression logistique est initialisé avec un équilibrage des classes (class_weight='balanced') pour gérer le déséquilibre dans les classes cibles. Une validation croisée k-fold est utilisée pour entraîner et évaluer le modèle, divisant les données en 5 parties (par défaut), afin de calculer des prédictions (predict) et des probabilités (predict_proba). Le modèle final est ensuite entraîné sur l'ensemble des données.
Sauvegarde du Modèle
Le modèle final est sauvegardé sous forme de fichier .pkl pour une utilisation future. Le nom du fichier inclut dynamiquement le nombre de folds pour refléter la configuration de la validation croisée.
Évaluation des Performances
Une matrice de confusion est calculée pour évaluer les performances du modèle en termes de classifications correctes et incorrectes pour chaque classe. Cette matrice est visualisée avec une heatmap de Seaborn. Des métriques telles que l'accuracy, la précision (precision) et la spécificité (specificity) sont calculées et affichées.
Courbe ROC et AUC
La courbe ROC est tracée pour illustrer la performance du modèle en termes de compromis entre les taux de faux positifs et de vrais positifs. L'aire sous la courbe (AUC) est calculée pour quantifier la capacité du modèle à différencier les classes.
Calcul du Taux de Pauvreté
Le taux de pauvreté réel est calculé à partir de la colonne cible (pcexp_binaire) en tenant compte des poids (hhweight) et des tailles des ménages (hhsize). Un taux prédit est également calculé en utilisant les prédictions du modèle. Ces valeurs sont imprimées pour comparaison.
Sauvegarde des Résultats
Enfin, les résultats finaux, y compris les prédictions, sont ajoutés au DataFrame initial et sauvegardés dans un fichier CSV dont le nom reflète le nombre de folds utilisés.
En Résumé
Le script combine plusieurs techniques d'analyse avancées : réduction de dimension, validation croisée, modélisation prédictive et visualisation, tout en tenant compte des déséquilibres dans les classes cibles et en assurant la traçabilité des résultats grâce à la sauvegarde des modèles, métriques et visualisations.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import seaborn as sns
import matplotlib.pyplot as plt
import os
import pickle

# Configuration des chemins
input_csv = r"D:\wealth_predict_sentinel\Data\processed_csv\EHCVM_2018_images_with_features_extracted_4096_fullyConv_sans_augm_couche_nongelee_batch_16_version_2.csv"
output_dir_csv = r"D:\wealth_predict_sentinel\Data\processed_csv"
output_dir_models = r"D:\wealth_predict_sentinel\models"
output_dir_figures = r"D:\wealth_predict_sentinel\figures_graphs"

# Chemins pour sauvegarder les modèles
scaler_path = os.path.join(output_dir_models, "scaler_fullyConv_sans_augm_couche_nongelee_batch_16_cross_validation_version_2.pkl")
pca_path = os.path.join(output_dir_models, "pca_fullyConv_sans_augm_couche_nongelee_batch_16_cross_validation_version_2.pkl")

# Chargement des données
df = pd.read_csv(input_csv)
features = [f'feature_{i}' for i in range(4096)]
target_binary = 'pcexp_binaire'

X = df[features]
y = df[target_binary]

# Paramètre dynamique pour k-fold
k = 5  # Nombre de folds pour la cross-validation

# Standardisation des données
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Sauvegarde du StandardScaler
with open(scaler_path, 'wb') as file:
    pickle.dump(scaler, file)

print(f"Le StandardScaler a été sauvegardé sous : {scaler_path}")

# Réduction de dimension avec PCA
pca = PCA(n_components=0.999)
X_pca = pca.fit_transform(X_scaled)
num_pca_components = X_pca.shape[1]
print(f"Nombre de composantes principales expliquant 99.9% de la variabilité : {num_pca_components}")

# Sauvegarde du modèle PCA
with open(pca_path, 'wb') as file:
    pickle.dump(pca, file)

print(f"Le modèle PCA a été sauvegardé sous : {pca_path}")

# Initialisation de la régression logistique
logistic_model = LogisticRegression(class_weight='balanced', max_iter=5000, solver='saga', random_state=42)

# k-Fold Cross-Validation
kf = KFold(n_splits=k, shuffle=True, random_state=42)
y_pred = cross_val_predict(logistic_model, X_pca, y, cv=kf, method="predict")
y_proba = cross_val_predict(logistic_model, X_pca, y, cv=kf, method="predict_proba")[:, 1]

# Entraînement final du modèle
logistic_model.fit(X_pca, y)

# Sauvegarde du modèle
logistic_model_path = os.path.join(output_dir_models, f"fullyConv_sans_augm_couche_nongelee_batch_16_logistic_pca_{k}-fold_logistic_reg_version_2.pkl")
with open(logistic_model_path, 'wb') as file:
    pickle.dump(logistic_model, file)

print(f"Le modèle a été sauvegardé sous : {logistic_model_path}")

# Évaluation des performances
conf_matrix = confusion_matrix(y, y_pred)
print("Matrice de confusion :")
print(conf_matrix)

# Affichage de la matrice de confusion avec seaborn
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=['Classe 0', 'Classe 1'], yticklabels=['Classe 0', 'Classe 1'])
plt.title(f"Matrice de confusion - {k}-Fold Cross-Validation")
plt.xlabel("Prédictions")
plt.ylabel("Valeurs Réelles")
conf_matrix_path = os.path.join(output_dir_figures, f"fullyConv_sans_augm_couche_nongelee_batch_16_conf_matrix_{k}-fold_logistic_reg_version_2.png")
plt.savefig(conf_matrix_path)
plt.show()

# Rapport de classification
print("Rapport de classification :")
print(classification_report(y, y_pred))

# Calcul des métriques
accuracy = (conf_matrix[0, 0] + conf_matrix[1, 1]) / conf_matrix.sum()
precision = conf_matrix[1, 1] / (conf_matrix[1, 1] + conf_matrix[0, 1])
specificity = conf_matrix[0, 0] / (conf_matrix[0, 0] + conf_matrix[0, 1])

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Specificity : {specificity:.4f}")

# Courbe ROC et AUC
fpr, tpr, _ = roc_curve(y, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'r--', label="Random Guessing")
plt.xlabel("Taux de faux positifs (FPR)")
plt.ylabel("Taux de vrais positifs (TPR)")
plt.title(f"ROC Curve - {k}-Fold Cross-Validation")
plt.legend()
plt.grid(True)
roc_curve_path = os.path.join(output_dir_figures, f"fullyConv_sans_augm_couche_nongelee_batch_16_roc_curve_{k}-fold_logistic_reg_version_2.png")
plt.savefig(roc_curve_path)
plt.show()

# Calcul du taux de pauvreté
df['pcexp_binaire_pred'] = y_pred

# Calcul du taux de pauvreté réel
weighted_real_poverty = (
    (df['hhweight'] * df['hhsize'] * df[target_binary]).sum() /
    (df['hhweight'] * df['hhsize']).sum()
)

# Calcul du taux de pauvreté prédit
weighted_predicted_poverty = (
    (df['hhweight'] * df['hhsize'] * df['pcexp_binaire_pred']).sum() /
    (df['hhweight'] * df['hhsize']).sum()
)

print(f"Taux de pauvreté réel : {weighted_real_poverty:.4%}")
print(f"Taux de pauvreté prédit : {weighted_predicted_poverty:.4%}")

# Sauvegarde des résultats finaux
final_csv_path = os.path.join(output_dir_csv, f"fullyConv_sans_augm_couche_nongelee_batch_16_test_with_poverty_{k}-fold_logistic_reg_version_2.csv")
df.to_csv(final_csv_path, index=False)

print(f"Les résultats ont été sauvegardés dans : {final_csv_path}")


# avec scikit-learn 1.6.1 pour gérer les problèmes d'incompatbilités dans la prévision

In [5]:
#!pip install -U scikit-learn==1.6.1


In [6]:
#import sklearn
#print(sklearn.__version__)  # Doit afficher 1.6.1


In [ ]:
'''
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import seaborn as sns
import matplotlib.pyplot as plt
import os
import pickle

# 🛠 Configuration des chemins
input_csv = r"D:\wealth_predict_sentinel\Data\processed_csv\EHCVM_2018_images_with_features_extracted_4096_fullyConv_sans_augm_couche_nongelee_batch_16.csv"
output_dir_csv = r"D:\wealth_predict_sentinel\Data\processed_csv"
output_dir_models = r"D:\wealth_predict_sentinel\models"
output_dir_figures = r"D:\wealth_predict_sentinel\figures_graphs"

# 🛠 Nouveaux chemins de sauvegarde (préfixe _v1.6 pour éviter les conflits)
scaler_path = os.path.join(output_dir_models, "scaler_fullyConv_sans_augm_couche_nongelee_batch_16_cross_validation_v1.6.pkl")
pca_path = os.path.join(output_dir_models, "pca_fullyConv_sans_augm_couche_nongelee_batch_16_cross_validation_v1.6.pkl")
logistic_model_path = os.path.join(output_dir_models, "fullyConv_sans_augm_couche_nongelee_batch_16_logistic_pca_5-fold_logistic_reg_v1.6.pkl")

# 🔹 Chargement des données
df = pd.read_csv(input_csv)
features = [f'feature_{i}' for i in range(4096)]
target_binary = 'pcexp_binaire'

X = df[features]
y = df[target_binary]

# 🔹 Paramètre pour la validation croisée
k = 5  # Nombre de folds

# 🔹 Standardisation des données
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 🛠 Sauvegarde du StandardScaler
with open(scaler_path, 'wb') as file:
    pickle.dump(scaler, file)

print(f"✅ StandardScaler sauvegardé sous : {scaler_path}")

# 🔹 Réduction de dimension avec PCA
pca = PCA(n_components=0.999)
X_pca = pca.fit_transform(X_scaled)
num_pca_components = X_pca.shape[1]
print(f"✅ Nombre de composantes principales expliquant 99.9% de la variabilité : {num_pca_components}")

# 🛠 Sauvegarde du modèle PCA
with open(pca_path, 'wb') as file:
    pickle.dump(pca, file)

print(f"✅ Modèle PCA sauvegardé sous : {pca_path}")

# 🔹 Initialisation de la régression logistique (scikit-learn 1.6.1)
logistic_model = LogisticRegression(class_weight='balanced', max_iter=5000, solver='saga', random_state=42)

# 🔹 k-Fold Cross-Validation
kf = KFold(n_splits=k, shuffle=True, random_state=42)
y_pred = cross_val_predict(logistic_model, X_pca, y, cv=kf, method="predict")
y_proba = cross_val_predict(logistic_model, X_pca, y, cv=kf, method="predict_proba")[:, 1]

# 🔹 Entraînement final du modèle
logistic_model.fit(X_pca, y)

# 🛠 Sauvegarde du modèle LogisticRegression
with open(logistic_model_path, 'wb') as file:
    pickle.dump(logistic_model, file)

print(f"✅ Modèle LogisticRegression sauvegardé sous : {logistic_model_path}")

# 🔹 Évaluation des performances
conf_matrix = confusion_matrix(y, y_pred)
print("🔹 Matrice de confusion :")
print(conf_matrix)

# 🔹 Affichage de la matrice de confusion avec seaborn
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=['Classe 0', 'Classe 1'], yticklabels=['Classe 0', 'Classe 1'])
plt.title(f"Matrice de confusion - {k}-Fold Cross-Validation")
plt.xlabel("Prédictions")
plt.ylabel("Valeurs Réelles")
conf_matrix_path = os.path.join(output_dir_figures, f"fullyConv_sans_augm_couche_nongelee_batch_16_conf_matrix_{k}-fold_logistic_reg_v1.6.png")
plt.savefig(conf_matrix_path)
plt.show()

# 🔹 Rapport de classification
print("🔹 Rapport de classification :")
print(classification_report(y, y_pred))

# 🔹 Calcul des métriques
accuracy = (conf_matrix[0, 0] + conf_matrix[1, 1]) / conf_matrix.sum()
precision = conf_matrix[1, 1] / (conf_matrix[1, 1] + conf_matrix[0, 1])
specificity = conf_matrix[0, 0] / (conf_matrix[0, 0] + conf_matrix[0, 1])

print(f"✅ Accuracy : {accuracy:.4f}")
print(f"✅ Precision : {precision:.4f}")
print(f"✅ Specificity : {specificity:.4f}")

# 🔹 Courbe ROC et AUC
fpr, tpr, _ = roc_curve(y, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'r--', label="Random Guessing")
plt.xlabel("Taux de faux positifs (FPR)")
plt.ylabel("Taux de vrais positifs (TPR)")
plt.title(f"ROC Curve - {k}-Fold Cross-Validation")
plt.legend()
plt.grid(True)
roc_curve_path = os.path.join(output_dir_figures, f"fullyConv_sans_augm_couche_nongelee_batch_16_roc_curve_{k}-fold_logistic_reg_v1.6.png")
plt.savefig(roc_curve_path)
plt.show()

# 🔹 Calcul du taux de pauvreté
df['pcexp_binaire_pred'] = y_pred

weighted_real_poverty = (
    (df['hhweight'] * df['hhsize'] * df[target_binary]).sum() /
    (df['hhweight'] * df['hhsize']).sum()
)

weighted_predicted_poverty = (
    (df['hhweight'] * df['hhsize'] * df['pcexp_binaire_pred']).sum() /
    (df['hhweight'] * df['hhsize']).sum()
)

print(f"✅ Taux de pauvreté réel : {weighted_real_poverty:.4%}")
print(f"✅ Taux de pauvreté prédit : {weighted_predicted_poverty:.4%}")

# 🛠 Sauvegarde des résultats finaux
final_csv_path = os.path.join(output_dir_csv, f"fullyConv_sans_augm_couche_nongelee_batch_16_test_with_poverty_{k}-fold_logistic_reg_v1.6.csv")
df.to_csv(final_csv_path, index=False)

print(f"✅ Les résultats ont été sauvegardés dans : {final_csv_path}")
'''


Ce script réalise une analyse statistique avancée pour évaluer la performance d'un modèle de prédiction de la pauvreté en utilisant des techniques de bootstrap. Voici son fonctionnement en détail :
Le script prend en entrée un jeu de données contenant des caractéristiques (features) extraites d'images et des informations sur la pauvreté des ménages. Il réalise une validation croisée avec bootstrap pour estimer de manière robuste les taux de pauvreté réels et prédits, tout en calculant des intervalles de confiance.
Le processus principal utilise une réduction de dimensionnalité (PCA) pour gérer les 4096 caractéristiques, puis applique une régression logistique pour la prédiction. Pour chaque itération bootstrap, il échantillonne les données avec remplacement, effectue l'entraînement et la validation croisée, puis calcule les taux de pauvreté pondérés en tenant compte du poids des ménages.
Le script optimise les ressources système en utilisant un traitement par lots et une parallélisation contrôlée (4 cœurs par défaut). Il surveille activement la consommation mémoire et inclut des mécanismes de nettoyage automatique pour éviter la surcharge. Les résultats finaux comprennent les taux moyens, les intervalles de confiance à 95%, et les erreurs standards, tous sauvegardés dans des fichiers CSV horodatés.
L'objectif final est de fournir une estimation statistiquement robuste de la précision du modèle de prédiction de la pauvreté, en tenant compte de l'incertitude dans les prédictions et en assurant que le processus peut s'exécuter efficacement même sur des machines avec des ressources limitées.

In [4]:
#!pip install psutil

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from joblib import Parallel, delayed
import gc
import os
from datetime import datetime
import logging
import warnings
import psutil

# Configuration du logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('bootstrap_process.log'),
        logging.StreamHandler()
    ]
)

class MemoryEfficientBootstrap:
    def __init__(self, input_path, output_dir, n_bootstrap=1000, n_cores=4, 
                 pca_components=0.999, cv_folds=5, batch_size=50):
        self.input_path = input_path
        self.output_dir = output_dir
        self.n_bootstrap = n_bootstrap
        self.n_cores = n_cores
        self.pca_components = pca_components
        self.cv_folds = cv_folds
        self.batch_size = batch_size
        self.features = None
        self.required_columns = ['hhweight', 'hhsize', 'pcexp_binaire']
        
    def load_and_validate_data(self):
        """Charge et valide les données d'entrée"""
        logging.info("Chargement des données...")
        try:
            df = pd.read_csv(self.input_path)
            
            # Validation des colonnes
            missing_cols = [col for col in self.required_columns if col not in df.columns]
            if missing_cols:
                raise ValueError(f"Colonnes manquantes: {missing_cols}")
                
            # Identification des features
            self.features = [col for col in df.columns if col.startswith('feature_')]
            if not self.features:
                raise ValueError("Aucune colonne feature_ trouvée")
            
            # Vérification des valeurs manquantes
            if df[self.required_columns].isnull().any().any():
                raise ValueError("Les colonnes nécessaires contiennent des valeurs manquantes.")
            
            logging.info(f"Données chargées avec succès: {len(df)} lignes")
            return df
        except Exception as e:
            logging.error(f"Erreur lors du chargement des données: {str(e)}")
            raise

    @staticmethod
    def calculate_poverty_rate(data):
        """Calcule les taux de pauvreté pondérés"""
        weights_size = data['hhweight'] * data['hhsize']
        real_rate = (weights_size * data['pcexp_binaire']).sum() / weights_size.sum()
        pred_rate = (weights_size * data['pcexp_binaire_pred']).sum() / weights_size.sum()
        return real_rate, pred_rate

    def process_single_bootstrap(self, df, seed):
        """Traite une seule itération bootstrap avec gestion de mémoire"""
        try:
            np.random.seed(seed)
            logging.info(f"Bootstrap itération {seed} démarrée...")
            
            # Échantillonnage bootstrap
            sample_indices = np.random.choice(len(df), size=len(df), replace=True)
            sample = df.iloc[sample_indices].copy()
            logging.info(f"Échantillon bootstrap {seed} généré.")
            
            # Prétraitement
            X_sample = sample[self.features].values
            y_sample = sample['pcexp_binaire'].values
            
            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X_sample)
            logging.info(f"Données normalisées pour l'itération {seed}.")
            
            pca = PCA(n_components=self.pca_components)
            X_pca = pca.fit_transform(X_scaled)
            logging.info(f"PCA appliqué pour l'itération {seed}.")
            
            # Validation croisée
            kf = KFold(n_splits=self.cv_folds, shuffle=True, random_state=seed)
            model = LogisticRegression(
                class_weight='balanced',
                max_iter=1000,
                solver='saga',
                random_state=seed
            )
            
            predictions = np.zeros(len(y_sample))
            for train_idx, val_idx in kf.split(X_pca):
                X_train, X_val = X_pca[train_idx], X_pca[val_idx]
                y_train = y_sample[train_idx]
                
                model_clone = clone(model)
                model_clone.fit(X_train, y_train)
                if not model_clone.n_iter_:
                    logging.error(f"Le modèle n'a pas convergé pour l'itération {seed}.")
                    return None
                
                predictions[val_idx] = model_clone.predict(X_val)
            
            sample['pcexp_binaire_pred'] = predictions
            rates = self.calculate_poverty_rate(sample)
            logging.info(f"Calcul des taux de pauvreté terminé pour l'itération {seed}.")
            
            return rates
        except Exception as e:
            logging.error(f"Erreur dans l'itération bootstrap {seed}: {str(e)}")
            return None

    def run_bootstrap_batches(self):
        """Exécute le bootstrap par lots pour optimiser la mémoire"""
        logging.info(f"Démarrage du processus bootstrap avec {self.n_cores} cœurs...")
        df = self.load_and_validate_data()
        
        all_results = []
        n_batches = (self.n_bootstrap + self.batch_size - 1) // self.batch_size
        
        for batch in range(n_batches):
            start_idx = batch * self.batch_size
            end_idx = min((batch + 1) * self.batch_size, self.n_bootstrap)
            batch_size = end_idx - start_idx
            
            logging.info(f"Traitement du lot {batch + 1}/{n_batches}")
            
            available_memory = psutil.virtual_memory().available / (1024 * 1024 * 1024)  # en GB
            logging.info(f"Mémoire disponible: {available_memory:.2f} GB")
            
            if available_memory < 2:  # Si moins de 2GB disponible
                logging.warning("Mémoire faible, pause de 10 secondes")
                import time
                time.sleep(10)
                gc.collect()
            
            seeds = range(start_idx, end_idx)
            
            batch_results = Parallel(n_jobs=self.n_cores)(
                delayed(self.process_single_bootstrap)(df, seed) 
                for seed in seeds
            )
            
            batch_results = [r for r in batch_results if r is not None]
            all_results.extend(batch_results)
            
            gc.collect()
            
            logging.info(f"Lot {batch + 1} terminé, {len(all_results)} résultats accumulés")
        
        if not all_results:
            logging.error("Aucun résultat bootstrap valide. Vérifiez les étapes précédentes.")
        
        return df, all_results

    def save_results(self, df, bootstrap_results):
        """Sauvegarde les résultats et les statistiques"""
        if not bootstrap_results:
            logging.error("Aucun résultat valide à sauvegarder.")
            return
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        bootstrap_rates_real, bootstrap_rates_pred = zip(*bootstrap_results)
        
        stats = {
            "real_rate": {
                "mean": np.mean(bootstrap_rates_real),
                "ci_lower": np.percentile(bootstrap_rates_real, 2.5),
                "ci_upper": np.percentile(bootstrap_rates_real, 97.5),
                "std_error": np.std(bootstrap_rates_real, ddof=1)
            },
            "pred_rate": {
                "mean": np.mean(bootstrap_rates_pred),
                "ci_lower": np.percentile(bootstrap_rates_pred, 2.5),
                "ci_upper": np.percentile(bootstrap_rates_pred, 97.5),
                "std_error": np.std(bootstrap_rates_pred, ddof=1)
            }
        }
        
        try:
            bootstrap_df = pd.DataFrame({
                'bootstrap_real': bootstrap_rates_real,
                'bootstrap_pred': bootstrap_rates_pred
            })
            bootstrap_df.to_csv(
                os.path.join(self.output_dir, f"bootstrap_results_{timestamp}.csv"),
                index=False
            )
            
            pd.DataFrame(stats).to_csv(
                os.path.join(self.output_dir, f"bootstrap_statistics_{timestamp}.csv")
            )
            
            for rate_type, values in stats.items():
                logging.info(f"\nStatistiques pour {rate_type}:")
                logging.info(f"Moyenne: {values['mean']:.4%}")
                logging.info(f"IC 95%: [{values['ci_lower']:.4%}, {values['ci_upper']:.4%}]")
                logging.info(f"Erreur standard: {values['std_error']:.4%}")
                
        except Exception as e:
            logging.error(f"Erreur lors de la sauvegarde des résultats: {str(e)}")
            raise

def main():
    # Configuration des chemins
    input_csv = r"D:\wealth_predict_sentinel\Data\processed_csv\EHCVM_2018_images_with_features_extracted_4096_fullyConv_sans_augm_couche_nongelee_batch_16_version_2.csv"
    output_dir = r"D:\wealth_predict_sentinel\Data\processed_csv"
    
    # Paramètres
    params = {
        'n_bootstrap': 1000,
        'n_cores': 8,  # Utilisation de 8 cœurs
        'pca_components': 0.999,
        'cv_folds': 5,
        'batch_size': 50  # Taille de lot réduite
    }
    
    try:
        bootstrap = MemoryEfficientBootstrap(input_csv, output_dir, **params)
        df, results = bootstrap.run_bootstrap_batches()
        bootstrap.save_results(df, results)
        logging.info("Processus terminé avec succès!")
    except Exception as e:
        logging.error(f"Erreur durant l'exécution: {str(e)}")
        raise

if __name__ == "__main__":
    main()
